### Step 1: Mount Drive & Verify Dataset

In [ ]:
import os
from google.colab import drive

In [ ]:
# Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ─ Your dataset path ─
# Based on your folder structure:
# MyDrive → ArchP → binary_dataset → positive & negative folders
POS_DIR = "/content/drive/MyDrive/binary_dataset/positive"
NEG_DIR = "/content/drive/MyDrive/binary_dataset/negative"

In [ ]:
# ── Scan both folders ─
VALID_EXT = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}

In [ ]:
def count_images(folder_path):
    """Count valid image files in a folder."""
    if not os.path.exists(folder_path):
        return None, []
    files = [
        f for f in os.listdir(folder_path)
        if os.path.splitext(f)[1].lower() in VALID_EXT
    ]
    return len(files), files

n_pos, pos_files = count_images(POS_DIR)
n_neg, neg_files = count_images(NEG_DIR)

In [ ]:
# ── Report ────────────────────────────────────────────────────────────────
print("=" * 45)
print("DATASET SCAN RESULTS")
print("=" * 45)

if n_pos is None:
    print(f" positive folder NOT found at:\n   {POS_DIR}")
else:
    print(f" Positive (geoglyph) : {n_pos} images")

if n_neg is None:
    print(f" negative folder NOT found at:\n   {NEG_DIR}")
else:
    print(f" Negative (background): {n_neg} images")

if n_pos and n_neg:
    print(f"\n   Total images  : {n_pos + n_neg}")
    print(f"   Class ratio   : {n_neg/n_pos:.1f}:1  (neg/pos)")
    print(f"\n   Sample positive filenames:")
    for f in pos_files[:3]:
        print(f"     {f}")
    print(f"\n   Sample negative filenames:")
    for f in neg_files[:3]:
        print(f"     {f}")

print("=" * 45)

DATASET SCAN RESULTS
 Positive (geoglyph) : 530 images
 Negative (background): 2181 images

   Total images  : 2711
   Class ratio   : 4.1:1  (neg/pos)

   Sample positive filenames:
     circle_geoglyph_circle_#acrc21.png
     circle_geoglyph_circle_#acdc5.png
     circle_geoglyph_circle_#acrc20.png

   Sample negative filenames:
     negative_amas21_-8.983_-67.021.png
     negative_amas15_-9.183_-67.480.png
     negative_amds1_-9.791_-66.820.png


### Step 2: Imports & Seed Setup

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from pathlib import Path
import json

In [ ]:
# ── Set all seeds for full reproducibility ──
# We set every random source: Python, NumPy, TensorFlow
# This ensures your results are identical every time you re-run
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
print("TensorFlow :", tf.__version__)
print("GPU        :", tf.config.list_physical_devices('GPU'))
print("Seeds set  : ✓")

TensorFlow : 2.19.0
GPU        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Seeds set  : ✓


### Step 3: Build Masster File & Stratified Split

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
POS_DIR    = "/content/drive/MyDrive/ArchP/binary_dataset/positive"
NEG_DIR    = "/content/drive/MyDrive/ArchP/binary_dataset/negative"

In [ ]:
# Using Colab local storage — does NOT count against your Drive quota
# Note: this resets when your Colab session ends, so we'll save only
# the final important files (splits CSV, best model weights) to Drive
OUTPUT_DIR = "/content/outputs"
IMG_SIZE   = 224    # standard input size for EfficientNet
BATCH_SIZE = 32
VALID_EXT  = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Output directory: /content/outputs


In [ ]:
# ── Build master dataframe ──
# Collect all image paths with their labels
# label 1 = geoglyph (positive), label 0 = background (negative)
pos_files = [(str(p), 1) for p in sorted(Path(POS_DIR).iterdir())
             if p.suffix.lower() in VALID_EXT]
neg_files = [(str(p), 0) for p in sorted(Path(NEG_DIR).iterdir())
             if p.suffix.lower() in VALID_EXT]

df = pd.DataFrame(pos_files + neg_files, columns=['filepath', 'label'])

print(f"\nTotal images : {len(df)}")
print(f"Positive     : {(df.label==1).sum()}")
print(f"Negative     : {(df.label==0).sum()}")


Total images : 2711
Positive     : 530
Negative     : 2181


In [ ]:
# ── Stratified Train / Val / Test split ─
# Stratified means each split maintains the same 4:1 neg/pos ratio
# as the full dataset — important for consistent evaluation
#
# We split ONCE and save to CSV — every model (EfficientNet-B0, B2,
# DenseNet, ViT, etc.) will use this exact same split so results
# are directly comparable across all architectures

# Step 1: carve out 15% as test set (never touched during training)
train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED
)

In [ ]:
# Step 2: from remaining 85%, take ~15% as validation
# (0.176 = 0.15/0.85 to get the correct final proportion)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.176, stratify=train_val_df['label'], random_state=SEED
)

In [ ]:
# Tag each row with its split name
train_df = train_df.copy(); train_df['split'] = 'train'
val_df   = val_df.copy();   val_df['split']   = 'val'
test_df  = test_df.copy();  test_df['split']  = 'test'

In [ ]:
# Combine and save to local storage
# This CSV is the single source of truth for all experiments
split_df  = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
split_csv = os.path.join(OUTPUT_DIR, "data_splits.csv")
split_df.to_csv(split_csv, index=False)

In [ ]:
# ── Compute class weights ──
# Our dataset is imbalanced (4:1). Class weights tell Keras to penalize
# missed geoglyphs (false negatives) more than false positives.
# Without this, the model can score ~80% accuracy by predicting
# "negative" for everything — which is useless for our task.
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_df['label'].values
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}


In [ ]:
# ── Summary ──
print("\nSPLIT SUMMARY")
print("=" * 55)
print(f"  {'Split':<8} {'Total':>6} {'Positive':>10} {'Negative':>10}")
print("-" * 55)
for name in ['train', 'val', 'test']:
    s    = split_df[split_df.split == name]
    npos = (s.label == 1).sum()
    nneg = (s.label == 0).sum()
    print(f"  {name:<8} {len(s):>6} {npos:>10} {nneg:>10}")
print("=" * 55)

print(f"\nClass weights:")
print(f"  Negative (0) : {class_weight_dict[0]:.4f}")
print(f"  Positive (1) : {class_weight_dict[1]:.4f}  ← geoglyphs penalized more")

print(f"\n✓ Split saved to  : {split_csv}")
print(f"✓ Free to proceed to Cell 4")


SPLIT SUMMARY
  Split     Total   Positive   Negative
-------------------------------------------------------
  train      1898        371       1527
  val         406         79        327
  test        407         80        327

Class weights:
  Negative (0) : 0.6215
  Positive (1) : 2.5580  ← geoglyphs penalized more

✓ Split saved to  : /content/outputs/data_splits.csv
✓ Free to proceed to Cell 4


### Why three splits?

- Train (70%) — the only data the model ever sees and learns from. Think of it as the textbook.
- Val (15%) — the model never trains on this, but we check performance on it after every epoch. This tells us when to stop training and whether the model is overfitting. Think of it as practice exams.
- Test (15%) — completely locked away until the very end. We evaluate here exactly once, after all training and tuning is done. This gives an honest, unbiased performance number that goes into your paper. Think of it as the final exam.

### Why is the ratio the same across all three?
Notice each split has roughly 4:1 negative to positive — same as the original dataset. This is what "stratified" means. If the test set had a different ratio it would give misleading results, so we preserve the natural distribution everywhere.

### Class weights

- Negative weight 0.62 — a wrong prediction on a background image is penalized less
- Positive weight 2.55 — a missed geoglyph is penalized ~4x more

This counterbalances the imbalance. Without it, the model could predict "background" for every single image and still get 80% accuracy — which is useless. The class weights force it to actually learn what a geoglyph looks like.

### Step 4: Build tf.data Datasets

In [ ]:
def load_and_preprocess(filepath, label):
    """
    Load one image chip from disk.

    Important: EfficientNet has its own normalization built in —
    it expects raw pixel values in [0, 255], NOT normalized to [0, 1].
    So we deliberately skip manual normalization here.
    """
    raw   = tf.io.read_file(filepath)
    image = tf.image.decode_image(raw, channels=3, expand_animations=False)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = tf.cast(image, tf.float32)  # keeps values in [0, 255]
    return image, tf.cast(label, tf.float32)

In [ ]:
def build_dataset(dataframe, shuffle=False):
    """
    Build a tf.data pipeline from a split dataframe.

    We use:
    - num_parallel_calls=AUTOTUNE : loads multiple images in parallel on CPU
    - prefetch=AUTOTUNE           : prepares next batch while GPU trains on current
    These two together keep the GPU fully utilized during training.
    """
    filepaths = dataframe['filepath'].values
    labels    = dataframe['label'].values.astype(np.float32)

    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))

    if shuffle:
        # reshuffle_each_iteration=True means different order each epoch
        # this is important — seeing images in a different order each epoch
        # helps the model generalize rather than memorize sequence patterns
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED,
                        reshuffle_each_iteration=True)

    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.ignore_errors() # ADDED LATER
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

In [ ]:
# Build the three splits
# shuffle=True only for training — val and test must always be in fixed order
train_ds = build_dataset(train_df, shuffle=True)
val_ds   = build_dataset(val_df,   shuffle=False)
test_ds  = build_dataset(test_df,  shuffle=False)
print("✓ Datasets rebuilt with ignore_errors")

✓ Datasets rebuilt with ignore_errors


In [ ]:
# # ── Verify shapes ─
# # Grab one batch and confirm dimensions are correct
# sample_images, sample_labels = next(iter(train_ds))

# print("Dataset pipeline built:")
# print(f"  Train : {len(train_df)} images → {len(train_ds)} batches")
# print(f"  Val   : {len(val_df)} images → {len(val_ds)} batches")
# print(f"  Test  : {len(test_df)} images → {len(test_ds)} batches")
# print(f"\nSample batch:")
# print(f"  Image shape : {sample_images.shape}  ← (batch, H, W, channels)")
# print(f"  Label shape : {sample_labels.shape}  ← (batch,)")
# print(f"  Pixel range : [{sample_images.numpy().min():.1f}, {sample_images.numpy().max():.1f}]  ← should be [0, 255]")
# print(f"  Labels seen : {sorted(set(sample_labels.numpy().astype(int)))}  ← should be [0, 1]")
# print(f"\n✓ Datasets ready")

In [ ]:
# Replace your print block with this
print("✓ Datasets rebuilt with ignore_errors")
print(f"  Train : {len(train_df)} images")
print(f"  Val   : {len(val_df)} images")
print(f"  Test  : {len(test_df)} images")
print(f"\nSample batch:")
sample_images, sample_labels = next(iter(train_ds))
print(f"  Image shape : {sample_images.shape}")
print(f"  Label shape : {sample_labels.shape}")
print(f"  Pixel range : [{sample_images.numpy().min():.1f}, {sample_images.numpy().max():.1f}]")
print(f"\n✓ Ready for Cell 5")

✓ Datasets rebuilt with ignore_errors
  Train : 1898 images
  Val   : 406 images
  Test  : 407 images

Sample batch:
  Image shape : (32, 224, 224, 3)
  Label shape : (32,)
  Pixel range : [0.0, 255.0]

✓ Ready for Cell 5


- We set BATCH_SIZE = 32, meaning we feed 32 images at a time to the model. So 1898 ÷ 32 = ~60 batches for training. The model goes through all 60 batches to complete one epoch. Val and test work the same way.

- This confirms one batch contains exactly 32 images, each resized to 224×224 pixels with 3 color channels (RGB). EfficientNet requires 224×224 — this is correct.

- Each of the 32 images has exactly one label — either 0 (background) or 1 (geoglyph). The shape (32,) means it's a flat list of 32 numbers, which is exactly what binary classification expects.

- Pixel values go from 0 to 255 — raw, unnormalized. This is intentional. EfficientNet handles its own normalization internally, so we deliberately did not normalize here. If we had normalized to [0,1] and EfficientNet normalized again, the values would be wrong and the pretrained weights would give garbage predictions.

- Confirms both classes are present in this batch — we're seeing both geoglyphs and background images. If only [0] showed up it would mean something went wrong with the shuffle or the label assignment.

Scan & Remove Corrupt Images

In [ ]:
# import os

# print("Scanning all images for corruption...")
# print("This may take 2-3 minutes...\n")

# corrupt_files = []

# for idx, row in split_df.iterrows():
#     filepath = row['filepath']
#     try:
#         # Read and decode the file using TensorFlow directly
#         raw   = tf.io.read_file(filepath)
#         image = tf.image.decode_image(raw, channels=3, expand_animations=False)
#         # Force execution — catches files that decode but give wrong shape
#         _ = image.numpy()
#     except Exception as e:
#         corrupt_files.append(filepath)
#         print(f"  ✗ Corrupt: {os.path.basename(filepath)}")
#         print(f"    Error  : {str(e)[:80]}")

# print(f"\n{'='*50}")
# print(f"Scan complete")
# print(f"  Total scanned : {len(split_df)}")
# print(f"  Clean         : {len(split_df) - len(corrupt_files)}")
# print(f"  Corrupt       : {len(corrupt_files)}")
# print(f"{'='*50}")

Run this to rebuild everything without the corrupt files:

In [ ]:
# # Only run this if corrupt files were found
# if corrupt_files:
#     print(f"\nRemoving {len(corrupt_files)} corrupt file(s) and rebuilding splits...")

#     # Drop corrupt rows from the master split dataframe
#     split_df_clean = split_df[~split_df['filepath'].isin(corrupt_files)].reset_index(drop=True)

#     # Rebuild the three split dataframes
#     train_df = split_df_clean[split_df_clean.split == 'train'].reset_index(drop=True)
#     val_df   = split_df_clean[split_df_clean.split == 'val'].reset_index(drop=True)
#     test_df  = split_df_clean[split_df_clean.split == 'test'].reset_index(drop=True)

#     # Rebuild datasets
#     train_ds = build_dataset(train_df, shuffle=True)
#     val_ds   = build_dataset(val_df,   shuffle=False)
#     test_ds  = build_dataset(test_df,  shuffle=False)

#     # Save the clean split to CSV (overwrites the old one)
#     split_df_clean.to_csv(os.path.join(OUTPUT_DIR, "data_splits.csv"), index=False)

#     print(f"\nRebuilt splits (corrupt files excluded):")
#     print(f"  {'Split':<8} {'Total':>6} {'Positive':>10} {'Negative':>10}")
#     print(f"  {'-'*38}")
#     for name in ['train', 'val', 'test']:
#         s    = split_df_clean[split_df_clean.split == name]
#         npos = (s.label == 1).sum()
#         nneg = (s.label == 0).sum()
#         print(f"  {name:<8} {len(s):>6} {npos:>10} {nneg:>10}")

#     print(f"\n✓ Datasets rebuilt — now re-run Cell 5 to rebuild the model")
#     print(f"  then re-run Cell 6 to train again")

# else:
#     print("\n✓ No corrupt files found — something else caused the error")
#     print("  Share the full error and we'll investigate")

**Above process is very slow so here is alternative**

### Copy Dataset to Colab Local Storage (fast) - still time consuming realized

In [ ]:
# import shutil

# print("Copying dataset from Drive to Colab local storage...")
# print("This is a one-time copy — scanning will be much faster after this\n")

# # Copy both folders to /content/ (local, not Drive)
# LOCAL_POS = "/content/binary_dataset/positive"
# LOCAL_NEG = "/content/binary_dataset/negative"

# # Remove if already exists (in case of partial copy)
# if os.path.exists("/content/binary_dataset"):
#     shutil.rmtree("/content/binary_dataset")

# shutil.copytree(POS_DIR, LOCAL_POS)
# shutil.copytree(NEG_DIR, LOCAL_NEG)

# n_pos = len(os.listdir(LOCAL_POS))
# n_neg = len(os.listdir(LOCAL_NEG))

# print(f"✓ Copied to local storage:")
# print(f"  Positive : {n_pos} images → {LOCAL_POS}")
# print(f"  Negative : {n_neg} images → {LOCAL_NEG}")
# print(f"\n✓ Now update paths to use local copies")

**FIXED HERE**
- You do not need to run that scan cell at all.
.ignore_errors() in the build_dataset function handles it automatically during training — any corrupt file gets silently skipped without crashing. The scan cell was just an attempt to find corrupt files manually, but this approach is cleaner and faster.

In [ ]:
# # Update paths back to Drive (we don't need local copy)
# POS_DIR = "/content/drive/MyDrive/ArchP/binary_dataset/positive"
# NEG_DIR = "/content/drive/MyDrive/ArchP/binary_dataset/negative"

# # Rebuild master dataframe from Drive paths
# pos_files = [(str(p), 1) for p in sorted(Path(POS_DIR).iterdir())
#              if p.suffix.lower() in VALID_EXT]
# neg_files = [(str(p), 0) for p in sorted(Path(NEG_DIR).iterdir())
#              if p.suffix.lower() in VALID_EXT]

# df = pd.DataFrame(pos_files + neg_files, columns=['filepath', 'label'])

# # Rebuild splits using the same saved CSV (splits don't change)
# split_df = pd.read_csv(os.path.join(OUTPUT_DIR, "data_splits.csv"))

# # Update filepaths in split_df to point to Drive
# # (in case they were pointing to /content/ from the failed copy)
# split_df['filepath'] = split_df['filepath'].str.replace(
#     '/content/binary_dataset',
#     '/content/drive/MyDrive/ArchP/binary_dataset',
#     regex=False
# )

# train_df = split_df[split_df.split == 'train'].reset_index(drop=True)
# val_df   = split_df[split_df.split == 'val'].reset_index(drop=True)
# test_df  = split_df[split_df.split == 'test'].reset_index(drop=True)

# def load_and_preprocess(filepath, label):
#     """
#     Load one image chip from disk.
#     EfficientNet expects raw [0,255] — no manual normalization.
#     """
#     raw   = tf.io.read_file(filepath)
#     image = tf.image.decode_image(raw, channels=3, expand_animations=False)
#     image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
#     image = tf.cast(image, tf.float32)
#     return image, tf.cast(label, tf.float32)

# def build_dataset(dataframe, shuffle=False):
#     """
#     Build tf.data pipeline with .ignore_errors() to silently skip
#     any corrupt or unreadable image files during training/validation.
#     """
#     filepaths = dataframe['filepath'].values
#     labels    = dataframe['label'].values.astype(np.float32)

#     ds = tf.data.Dataset.from_tensor_slices((filepaths, labels))

#     if shuffle:
#         ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED,
#                         reshuffle_each_iteration=True)

#     ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

#     # This is the key fix — any corrupt file that causes a decode error
#     # is silently skipped instead of crashing the entire training run
#     ds = ds.ignore_errors()

#     ds = ds.batch(BATCH_SIZE)
#     ds = ds.prefetch(tf.data.AUTOTUNE)
#     return ds

# train_ds = build_dataset(train_df, shuffle=True)
# val_ds   = build_dataset(val_df,   shuffle=False)
# test_ds  = build_dataset(test_df,  shuffle=False)

# print("✓ Datasets rebuilt with error handling")
# print(f"  Train : {len(train_df)} images")
# print(f"  Val   : {len(val_df)} images")
# print(f"  Test  : {len(test_df)} images")
# print(f"\n✓ Corrupt files will be silently skipped during training")
# print(f"  Now re-run Cell 5 to rebuild the model, then Cell 6 to train")

### Step 5: Build EfficientNet-B0 Model

In [ ]:
def build_efficientnet_b0(trainable_backbone=False):
    """
    EfficientNet-B0 for binary geoglyph classification.

    We use a two-stage training strategy:
      Stage 1 (this cell) → backbone FROZEN, only the head trains
                            Fast and stable. Protects the pretrained
                            ImageNet features from being destroyed by
                            large random gradients from our new head.

      Stage 2 (later)     → backbone UNFROZEN, full model fine-tunes
                            at a very small learning rate. Gently adapts
                            the backbone features to satellite imagery.

    Why EfficientNet-B0?
      - Only 5.3M parameters → low overfitting risk on 371 training positives
      - Compound scaling balances depth, width and resolution efficiently
      - Strong ImageNet features transfer well to aerial/satellite imagery
    """

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_image")

    # ── Augmentation block ────────────────────────────────────────────────
    # Lives inside the model graph — Keras automatically applies it
    # during training and skips it during validation and inference.
    # No need to manage this manually.
    #
    # Why these specific augmentations?
    #   - Flip horizontal+vertical : geoglyphs have no canonical orientation
    #     in satellite imagery — a circle looks the same from any direction
    #   - Rotation ±90°            : squares and rectangles can appear at
    #     any compass angle
    #   - Brightness/Contrast ±15% : handles lighting variation across chips
    #     captured on different days or atmospheric conditions
    #   - Zoom ±10%                : handles minor scale differences
    #   - NO elastic/perspective   : would destroy precise geometric shapes
    #     (circles, parallelograms) that define geoglyphs
    x = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(factor=0.25, fill_mode='reflect'),
        layers.RandomBrightness(factor=0.15),
        layers.RandomContrast(factor=0.15),
        layers.RandomZoom(height_factor=(-0.1, 0.1), fill_mode='reflect'),
    ], name="augmentation")(inputs)

    # ── EfficientNet-B0 backbone ──────────────────────────────────────────
    # include_top=False  → removes the original 1000-class ImageNet head
    # weights='imagenet' → loads pretrained weights (much better than
    #                      random initialization on a small dataset)
    #
    # EfficientNet normalizes inputs internally from [0,255] → [-1,1]
    # This is why we kept pixel values in [0,255] in the data pipeline
    backbone = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_tensor=x
    )

    # Freeze backbone — all 237 backbone layers are locked in Stage 1
    backbone.trainable = trainable_backbone

    # ── Classification head ───────────────────────────────────────────────
    # GlobalAveragePooling2D:
    #   Converts the backbone's final feature map (7×7×1280)
    #   into a flat 1280-dimensional vector by averaging spatially.
    #   Better than Flatten for small datasets — far fewer parameters.
    x = layers.GlobalAveragePooling2D(name="gap")(backbone.output)

    # BatchNormalization:
    #   Normalizes activations before the dense layers.
    #   Especially important in Stage 1 because the backbone is frozen
    #   and only the head is updating — BN keeps training stable.
    x = layers.BatchNormalization(name="head_bn")(x)

    # Dense(256):
    #   Learns task-specific combinations of backbone features.
    #   256 units is appropriate for binary classification —
    #   large enough to learn, small enough not to overfit.
    x = layers.Dense(256, activation='relu', name="head_dense")(x)

    # Dropout(0.4):
    #   Randomly zeros 40% of neurons during training.
    #   Our primary defense against overfitting on only 371 positives.
    x = layers.Dropout(0.4, name="head_dropout")(x)

    # Output:
    #   Single neuron with sigmoid → outputs P(geoglyph) in [0, 1]
    #   Threshold at 0.5: above = geoglyph, below = background
    outputs = layers.Dense(1, activation='sigmoid', name="output")(x)

    model = keras.Model(inputs=inputs, outputs=outputs,
                        name="EfficientNetB0_geoglyph")
    return model

In [ ]:
# Build with frozen backbone (Stage 1)
model = build_efficientnet_b0(trainable_backbone=False)

# ── Verify parameter counts ───────────────────────────────────────────────
total_params     = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
frozen_params    = total_params - trainable_params

print("=" * 50)
print("MODEL BUILT — EfficientNet-B0")
print("=" * 50)
print(f"  Total parameters    : {total_params:>10,}")
print(f"  Trainable (head)    : {trainable_params:>10,}  ← updates in Stage 1")
print(f"  Frozen (backbone)   : {frozen_params:>10,}  ← locked")
print(f"  Trainable ratio     : {trainable_params/total_params*100:.1f}%")

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
MODEL BUILT — EfficientNet-B0
  Total parameters    :  4,382,884
  Trainable (head)    :    330,753  ← updates in Stage 1
  Frozen (backbone)   :  4,052,131  ← locked
  Trainable ratio     : 7.5%


In [ ]:
# ── Quick forward pass test ───────────────────────────────────────────────
# Run 4 dummy images through the model before training anything.
# If shapes are wrong we catch it here, not mid-training.
dummy_out = model(tf.zeros([4, IMG_SIZE, IMG_SIZE, 3]), training=False)
print(f"\nForward pass test:")
print(f"  Output shape : {dummy_out.shape}  ← should be (4, 1)")
print(f"  Output range : [{dummy_out.numpy().min():.4f}, {dummy_out.numpy().max():.4f}]  ← should be in [0, 1]")

if dummy_out.shape == (4, 1):
    print("\n✓ Model ready — proceed to Cell 6 (compile + train)")
else:
    print("\n✗ Shape mismatch — something is wrong with the architecture")


Forward pass test:
  Output shape : (4, 1)  ← should be (4, 1)
  Output range : [0.4129, 0.4129]  ← should be in [0, 1]

✓ Model ready — proceed to Cell 6 (compile + train)


- Downloading the weights — EfficientNet automatically downloaded pretrained ImageNet weights (16MB) from Google's servers. This happens only once per session.
- Total 4.38M, Trainable 330K (7.5%) — The backbone's 4M parameters are completely locked. Only our custom head's 330K parameters will update during Stage 1. This is exactly right.
- Output range [0.4129, 0.4129] — All four dummy images gave the same probability (~41% chance of being a geoglyph). They're identical because we fed zeros, so identical input gives identical output. The value being in [0,1] is what matters — sigmoid is working correctly.

### Step 6: Compile & Train Stage 1 (Frozen Backbone)

In [ ]:
# ── Compile the model ─────────────────────────────────────────────────────
# Three things to define:
#
# Optimizer — Adam with lr=1e-3
#   A relatively high learning rate is fine here because only the small
#   head is training. The backbone is frozen so there's no risk of
#   destroying the pretrained features with large updates.
#
# Loss — BinaryCrossentropy
#   Standard loss for binary classification. Works with sigmoid output.
#   label_smoothing=0.1 slightly softens the targets (0→0.1, 1→0.9)
#   This prevents the model from becoming overconfident, which helps
#   generalization on a small dataset like ours.
#
# Metrics — we track four things:
#   Accuracy  : overall correct predictions (can be misleading with imbalance)
#   AUC-ROC   : most important metric — measures separability between classes
#               regardless of threshold. 0.5 = random, 1.0 = perfect.
#               This is the primary metric for our paper.
#   Precision : of all predicted geoglyphs, how many actually are?
#   Recall    : of all actual geoglyphs, how many did we catch?
#               Recall is critical — a missed geoglyph is worse than
#               a false alarm for Walker Lab's discovery pipeline.

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.BinaryAccuracy(name='accuracy'),
        keras.metrics.AUC(name='auc_roc', curve='ROC'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)
print("✓ Model compiled")

✓ Model compiled


In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────────────
# Callbacks are functions that run automatically after each epoch.
# We use three:

# 1. ModelCheckpoint
#    Saves model weights only when val_auc_roc improves.
#    This means we always keep the BEST model, not the last one.
#    If the model starts overfitting in later epochs, we still have
#    the best checkpoint saved.
# checkpoint_path = os.path.join(OUTPUT_DIR, "efficientnetb0_stage1_best.weights.h5")
checkpoint_path = "/content/drive/MyDrive/ArchP/efficientnetb0_stage1_best.weights.h5"

checkpoint_cb = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_auc_roc',       # watch validation AUC-ROC
    mode='max',                  # higher AUC is better
    save_best_only=True,         # only save when it improves
    save_weights_only=True,      # saves disk space vs full model
    verbose=1
)

# 2. EarlyStopping
#    Stops training automatically if val_auc_roc does not improve
#    for 7 consecutive epochs (patience=7).
#    This prevents wasted computation and overfitting.
#    restore_best_weights=True means when training stops, the model
#    automatically rolls back to its best checkpoint.
earlystop_cb = keras.callbacks.EarlyStopping(
    monitor='val_auc_roc',
    mode='max',
    patience=7,                  # wait 7 epochs before stopping
    restore_best_weights=True,
    verbose=1
)

# 3. ReduceLROnPlateau
#    If val_auc_roc stops improving for 3 epochs, cut the learning
#    rate by half. This helps the model escape flat regions and
#    fine-tune more carefully as it gets closer to convergence.
reducelr_cb = keras.callbacks.ReduceLROnPlateau(
    monitor='val_auc_roc',
    mode='max',
    factor=0.5,                  # new_lr = current_lr × 0.5
    patience=3,
    min_lr=1e-6,                 # never go below this
    verbose=1
)

print("✓ Callbacks ready")
print(f"  Best weights will be saved to: {checkpoint_path}")

✓ Callbacks ready
  Best weights will be saved to: /content/drive/MyDrive/ArchP/efficientnetb0_stage1_best.weights.h5


In [ ]:
# ── Train Stage 1 (frozen backbone) ──────────────────────────────────────
# We run up to 30 epochs but EarlyStopping will likely stop it earlier.
# class_weight tells Keras to penalize missed geoglyphs more heavily —
# this is the class_weight_dict we computed in Cell 3.

print("\nStarting Stage 1 training (frozen backbone)...")
print("Watching val_auc_roc — higher is better\n")

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=[checkpoint_cb, earlystop_cb, reducelr_cb],
    verbose=1
)

print("\n✓ Stage 1 training complete")


Starting Stage 1 training (frozen backbone)...
Watching val_auc_roc — higher is better

Epoch 1/30
     60/Unknown 91s 1s/step - accuracy: 0.6039 - auc_roc: 0.6474 - loss: 0.8759 - precision: 0.2641 - recall: 0.5859

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 1: val_auc_roc improved from -inf to 0.65652, saving model to /content/drive/MyDrive/ArchP/efficientnetb0_stage1_best.weights.h5
60/60 ━━━━━━━━━━━━━━━━━━━━ 115s 2s/step - accuracy: 0.6042 - auc_roc: 0.6477 - loss: 0.8759 - precision: 0.2643 - recall: 0.5862 - val_accuracy: 0.7834 - val_auc_roc: 0.6565 - val_loss: 0.5572 - val_precision: 0.4516 - val_recall: 0.3733 - learning_rate: 0.0010
Epoch 2/30
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 539ms/step - accuracy: 0.6357 - auc_roc: 0.7220 - loss: 0.7946 - precision: 0.3079 - recall: 0.6708
Epoch 2: val_auc_roc improved from 0.65652 to 0.67844, saving model to /content/drive/MyDrive/ArchP/efficientnetb0_stage1_best.weights.h5
60/60 ━━━━━━━━━━━━━━━━━━━━ 40s 668ms/step - accuracy: 0.6361 - auc_roc: 0.7223 - loss: 0.7940 - precision: 0.3082 - recall: 0.6709 - val_accuracy: 0.7781 - val_auc_roc: 0.6784 - val_loss: 0.5421 - val_precision: 0.4259 - val_recall: 0.3067 - learning_rate: 0.0010
Epoch 3/30
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 518ms/step - accur

### Step 7: Unfreeze Backbone & Fine-Tune

In [ ]:
# ── Load best Stage 1 weights into fresh model ────────────────────────────
model.load_weights(checkpoint_path)
print("✓ Best Stage 1 weights loaded")

# ── Unfreeze all layers EXCEPT BatchNormalization ─────────────────────────
# BatchNormalization layers have running statistics built during Stage 1.
# If we unfreeze them they reset and destroy learned representations.
# Keeping BN frozen while unfreezing everything else gives stable fine-tuning.
for layer in model.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False  # keep BN statistics frozen
    else:
        layer.trainable = True   # unfreeze conv + dense layers

# Count parameters
total_params     = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
frozen_params    = total_params - trainable_params

print(f"\nStage 2 parameter status:")
print(f"  Total parameters    : {total_params:>10,}")
print(f"  Trainable           : {trainable_params:>10,}  ← conv + dense layers")
print(f"  Frozen (BN only)    : {frozen_params:>10,}  ← BN stats preserved")

# ── Recompile with very small learning rate ───────────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=[
        keras.metrics.BinaryAccuracy(name='accuracy'),
        keras.metrics.AUC(name='auc_roc', curve='ROC'),
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
    ]
)
print("\n✓ Model recompiled with lr=1e-5")

# ── Callbacks — checkpoint saves directly to Drive ────────────────────────
checkpoint_path_s2 = "/content/drive/MyDrive/ArchP/efficientnetb0_stage2_best.weights.h5"

checkpoint_cb_s2 = keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path_s2,
    monitor='val_auc_roc',
    mode='max',
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)

earlystop_cb_s2 = keras.callbacks.EarlyStopping(
    monitor='val_auc_roc',
    mode='max',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reducelr_cb_s2 = keras.callbacks.ReduceLROnPlateau(
    monitor='val_auc_roc',
    mode='max',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

print(f"\n✓ Callbacks ready")
print(f"  Best weights will be saved to: {checkpoint_path_s2}")
print("\nStarting Stage 2 (BN frozen, all other layers fine-tuning)...")
print("Epoch 1 val_auc_roc should start above 0.75\n")

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    class_weight=class_weight_dict,
    callbacks=[checkpoint_cb_s2, earlystop_cb_s2, reducelr_cb_s2],
    verbose=1
)

print("\n✓ Stage 2 training complete")

✓ Best Stage 1 weights loaded

Stage 2 parameter status:
  Total parameters    :  4,382,884
  Trainable           :  4,293,725  ← conv + dense layers
  Frozen (BN only)    :     89,159  ← BN stats preserved

✓ Model recompiled with lr=1e-5

✓ Callbacks ready
  Best weights will be saved to: /content/drive/MyDrive/ArchP/efficientnetb0_stage2_best.weights.h5

Starting Stage 2 (BN frozen, all other layers fine-tuning)...
Epoch 1 val_auc_roc should start above 0.75

Epoch 1/30
     60/Unknown 55s 518ms/step - accuracy: 0.8053 - auc_roc: 0.8930 - loss: 0.4760 - precision: 0.4933 - recall: 0.8023
Epoch 1: val_auc_roc improved from -inf to 0.77012, saving model to /content/drive/MyDrive/ArchP/efficientnetb0_stage2_best.weights.h5
60/60 ━━━━━━━━━━━━━━━━━━━━ 65s 698ms/step - accuracy: 0.8053 - auc_roc: 0.8930 - loss: 0.4760 - precision: 0.4934 - recall: 0.8024 - val_accuracy: 0.7727 - val_auc_roc: 0.7701 - val_loss: 0.5419 - val_precision: 0.4468 - val_recall: 0.5600 - learning_rate: 1.0000e-05

In [ ]:
# ── Load best Stage 2 weights ─────────────────────────────────────────────
model.load_weights("/content/drive/MyDrive/ArchP/efficientnetb0_stage2_best.weights.h5")
print("✓ Best Stage 2 weights loaded")

# ── Evaluate on test set ──────────────────────────────────────────────────
# This is the FIRST time the model sees test data.
# This number is the one that goes into your paper.
print("\nEvaluating on held-out test set...")
results = model.evaluate(test_ds, verbose=1)

metrics = ['loss', 'accuracy', 'auc_roc', 'precision', 'recall']
print(f"\n{'='*45}")
print(f"  EFFICIENTNET-B0 — FINAL TEST RESULTS")
print(f"{'='*45}")
for name, val in zip(metrics, results):
    print(f"  {name:<12} : {val:.4f}")
print(f"{'='*45}")
print(f"\n  F1 Score : {2 * results[4] * results[3] / (results[4] + results[3]):.4f}")

✓ Best Stage 2 weights loaded

Evaluating on held-out test set...
13/13 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.8693 - auc_roc: 0.8535 - loss: 0.4337 - precision: 0.7204 - recall: 0.5034

  EFFICIENTNET-B0 — FINAL TEST RESULTS
  loss         : 0.4187
  accuracy     : 0.8722
  auc_roc      : 0.8813
  precision    : 0.7121
  recall       : 0.5875

  F1 Score : 0.6438


What these numbers mean:

- AUC-ROC 0.88 — strong discriminative ability. The model correctly separates geoglyphs from background 88% of the time regardless of threshold. This is your headline number for the paper.
- Accuracy 87.2% — solid, especially given the 4:1 class imbalance
- Precision 0.71 — when the model predicts geoglyph, it is correct 71% of the time
- Recall 0.59 — the model catches 59% of actual geoglyphs. This is the weakest point — it is missing ~41% of geoglyphs
- F1 0.64 — harmonic mean of precision and recall. Reflects the precision/recall tradeoff